In [ ]:
pip install seaborn

In [ ]:
import pandas as pd 

In [ ]:
df = pd.read_csv("train_data.csv")

In [ ]:
df.head()

In [ ]:
corMat = df.corr()
corMat

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random

def run_quick_eda(
    df: pd.DataFrame, 
    target: str = 'bad_flag',
    prefix_for_aggregation: str = 'onus_attributes',  # example prefix
    sample_corr_features: int = 50  # how many features to sample for correlation
):
    """
    Perform a quick end-to-end EDA pipeline:
      1. Basic dataset info
      2. Missing data overview
      3. Target distribution
      4. Numeric summary
      5. Distribution plots for a random sample of numeric columns
      6. Correlation matrix on a sample of numeric columns
      7. (Optional) Example of prefix-based aggregation
    """
    
    print("\n=== 1. BASIC DATASET INFO ===")
    print("Shape of Data:", df.shape)
    print("\nData Types and Memory Usage:")
    print(df.info(memory_usage=True))
    
    # 2. Missing data overview
    print("\n=== 2. MISSING DATA OVERVIEW ===")
    missing_counts = df.isnull().sum()
    missing_percent = (missing_counts / len(df)) * 100
    missing_summary = pd.DataFrame({
        'MissingCount': missing_counts,
        'MissingPercent': missing_percent
    }).sort_values('MissingPercent', ascending=False)
    print("Missing Values per Column (Top 10):")
    print(missing_summary.head(10))
    print(f"\nNumber of columns with any missing values: {sum(missing_counts > 0)}")
    
    # 3. Target distribution
    if target in df.columns:
        print(f"\n=== 3. TARGET ('{target}') DISTRIBUTION ===")
        print(df[target].value_counts(dropna=False))
        print("Percentage distribution:")
        print(df[target].value_counts(normalize=True, dropna=False)*100)
        
        # Quick bar plot of target distribution
        plt.figure(figsize=(4,4))
        sns.countplot(x=target, data=df)
        plt.title(f"Target Distribution: {target}")
        plt.show()
    else:
        print(f"\nTarget column '{target}' not found in DataFrame.")
    
    # 4. Numeric summary
    print("\n=== 4. BASIC NUMERIC SUMMARY ===")
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        print(df[numeric_cols].describe().T.head(15))  # show first 15 numeric columns
    else:
        print("No numeric columns found.")
    
    # 5. Distribution plots for random sample of numeric columns
    print("\n=== 5. DISTRIBUTION PLOTS (RANDOM SAMPLE) ===")
    num_to_plot = min(5, len(numeric_cols))  # plot up to 5 randomly
    if num_to_plot > 0:
        sample_numeric_cols = random.sample(numeric_cols, num_to_plot)
        df[sample_numeric_cols].hist(figsize=(15, 8), bins=30)
        plt.suptitle("Histograms of Random Sample of Numeric Columns", y=1.02)
        plt.show()
    else:
        print("No numeric columns to plot.")
    
    # 6. Correlation matrix on a sample of numeric columns
    print("\n=== 6. CORRELATION MATRIX (SAMPLE) ===")
    if len(numeric_cols) > 1:
        sample_size = min(sample_corr_features, len(numeric_cols))
        corr_cols = random.sample(numeric_cols, sample_size)
        corr_matrix = df[corr_cols].corr()
        
        plt.figure(figsize=(12, 10))
        sns.heatmap(corr_matrix, cmap='coolwarm', annot=False)
        plt.title(f"Correlation Matrix for {sample_size} Sampled Numeric Features")
        plt.show()
    else:
        print("Not enough numeric columns to create a correlation matrix.")
    
    # 7. Optional: Prefix-based aggregation example
    print(f"\n=== 7. OPTIONAL PREFIX AGGREGATION: '{prefix_for_aggregation}' ===")
    prefix_cols = [col for col in df.columns if col.startswith(prefix_for_aggregation)]
    if prefix_cols:
        # Example: create an aggregated feature for the prefix
        df[f'{prefix_for_aggregation}_mean'] = df[prefix_cols].mean(axis=1)
        print(f"Created '{prefix_for_aggregation}_mean' as mean of {len(prefix_cols)} columns.")
        
        # Plot distribution of the new aggregated column
        plt.figure(figsize=(6,4))
        sns.histplot(df[f'{prefix_for_aggregation}_mean'], bins=30)
        plt.title(f"Distribution of {prefix_for_aggregation}_mean")
        plt.show()
    else:
        print(f"No columns found with prefix '{prefix_for_aggregation}'.")
    
    print("\n=== EDA COMPLETED ===")


# Usage Example (assuming you already loaded your data into df):
# df = pd.read_csv('train_data.csv')
# run_quick_eda(df, target='bad_flag')
